# TF-IDF + ClassifierChain

Notebook nay dung `ClassifierChain` cho bai toan multi-label aspect detection. Feature dau vao la TF-IDF word n-gram ket hop character n-gram. Base classifier la Logistic Regression de co `predict_proba` phuc vu tune threshold.

In [ ]:
import inspect

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import ClassifierChain
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, hamming_loss, multilabel_confusion_matrix

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

RANDOM_STATE = 42

## 1. Load data

In [ ]:
df = pd.read_csv('Data/Restaurant_ABSA_processed.csv')

aspect_cols = ['food', 'service', 'price', 'ambiance', 'miscellaneous']
text_col = 'review_cleaned'

df[text_col] = df[text_col].fillna('')
X = df[text_col]
y = df[aspect_cols].astype(int)

print(df.shape)
display(df.head())
display(y.sum().sort_values(ascending=False).rename('positive_count').to_frame())

## 2. Train / validation / test split

Validation set duoc dung de tune threshold. Test set chi dung cho danh gia cuoi cung.

In [ ]:
def stratify_if_possible(labels):
    counts = pd.Series(labels).value_counts()
    if counts.min() >= 2:
        return labels
    return None


label_combo = y.astype(str).agg(''.join, axis=1)

X_train_val, X_test, y_train_val, y_test, combo_train_val, combo_test = train_test_split(
    X,
    y,
    label_combo,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_if_possible(label_combo),
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_if_possible(combo_train_val),
)

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

display(pd.DataFrame({
    'train': y_train.sum(),
    'validation': y_val.sum(),
    'test': y_test.sum(),
}))

## 3. Tao model ClassifierChain

`ClassifierChain` hoc lan luot tung nhan. Du doan cua nhan truoc se duoc dung lam feature bo sung cho nhan sau, nen model co the hoc quan he giua cac aspect.

In [ ]:
def make_classifier_chain(base_estimator, order='random', random_state=RANDOM_STATE):
    chain_params = {
        'order': order,
        'random_state': random_state,
    }

    # sklearn versions differ: newer versions use estimator, older ones use base_estimator.
    if 'estimator' in inspect.signature(ClassifierChain).parameters:
        chain_params['estimator'] = base_estimator
    else:
        chain_params['base_estimator'] = base_estimator

    return ClassifierChain(**chain_params)


def build_chain_model(random_state=RANDOM_STATE):
    word_tfidf = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 3),
        max_features=8000,
        min_df=1,
        sublinear_tf=True,
    )

    char_tfidf = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=12000,
        min_df=1,
        sublinear_tf=True,
    )

    base_estimator = LogisticRegression(
        C=2.0,
        class_weight='balanced',
        max_iter=2000,
        solver='liblinear',
        random_state=random_state,
    )

    return Pipeline([
        ('features', FeatureUnion([
            ('word_tfidf', word_tfidf),
            ('char_tfidf', char_tfidf),
        ])),
        ('clf', make_classifier_chain(
            base_estimator=base_estimator,
            order='random',
            random_state=random_state,
        )),
    ])

## 4. Train single ClassifierChain

In [ ]:
single_chain = build_chain_model(random_state=RANDOM_STATE)
single_chain.fit(X_train, y_train)

print('Learned chain order:', single_chain.named_steps['clf'].order_)
print('Aspect order:', [aspect_cols[i] for i in single_chain.named_steps['clf'].order_])

single_chain

## 5. Tune threshold tren validation set

In [ ]:
def tune_thresholds(y_true, proba, aspect_cols, grid=None):
    if grid is None:
        grid = np.arange(0.10, 0.91, 0.01)

    y_true = np.asarray(y_true)
    thresholds = {}
    rows = []

    for i, aspect in enumerate(aspect_cols):
        best_threshold = 0.5
        best_f1 = -1.0

        for threshold in grid:
            pred = (proba[:, i] >= threshold).astype(int)
            f1 = f1_score(y_true[:, i], pred, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold

        thresholds[aspect] = best_threshold
        rows.append({
            'aspect': aspect,
            'threshold': best_threshold,
            'validation_f1': best_f1,
        })

    return thresholds, pd.DataFrame(rows)


def predict_with_thresholds(proba, thresholds, aspect_cols, top_k=None, ensure_one_label=True):
    pred = np.zeros(proba.shape, dtype=int)

    for i, aspect in enumerate(aspect_cols):
        pred[:, i] = (proba[:, i] >= thresholds[aspect]).astype(int)

    if ensure_one_label:
        empty_rows = np.where(pred.sum(axis=1) == 0)[0]
        if len(empty_rows) > 0:
            pred[empty_rows, np.argmax(proba[empty_rows], axis=1)] = 1

    if top_k is not None:
        for row_idx in range(pred.shape[0]):
            if pred[row_idx].sum() > top_k:
                keep = np.argsort(proba[row_idx])[-top_k:]
                pred[row_idx] = 0
                pred[row_idx, keep] = 1

    return pred


val_proba = single_chain.predict_proba(X_val)
thresholds, threshold_report = tune_thresholds(y_val.values, val_proba, aspect_cols)

display(threshold_report)

## 6. Danh gia single chain

In [ ]:
def evaluate_multilabel(y_true, y_pred, aspect_cols, title='Evaluation'):
    print(title)
    print('-' * len(title))
    print(classification_report(y_true, y_pred, target_names=aspect_cols, zero_division=0))
    print('F1 Micro:', f1_score(y_true, y_pred, average='micro', zero_division=0))
    print('F1 Macro:', f1_score(y_true, y_pred, average='macro', zero_division=0))
    print('F1 Samples:', f1_score(y_true, y_pred, average='samples', zero_division=0))
    print('Hamming Loss:', hamming_loss(y_true, y_pred))


y_val_pred_default = (val_proba >= 0.5).astype(int)
y_val_pred_tuned = predict_with_thresholds(val_proba, thresholds, aspect_cols)
y_val_pred_tuned_top2 = predict_with_thresholds(val_proba, thresholds, aspect_cols, top_k=2)

evaluate_multilabel(y_val.values, y_val_pred_default, aspect_cols, title='Validation - default threshold 0.5')
print('\n')
evaluate_multilabel(y_val.values, y_val_pred_tuned, aspect_cols, title='Validation - tuned thresholds')
print('\n')
evaluate_multilabel(y_val.values, y_val_pred_tuned_top2, aspect_cols, title='Validation - tuned thresholds + top_k=2')

In [ ]:
test_proba = single_chain.predict_proba(X_test)

y_test_pred_default = (test_proba >= 0.5).astype(int)
y_test_pred_tuned = predict_with_thresholds(test_proba, thresholds, aspect_cols)
y_test_pred_tuned_top2 = predict_with_thresholds(test_proba, thresholds, aspect_cols, top_k=2)

evaluate_multilabel(y_test.values, y_test_pred_default, aspect_cols, title='Test - default threshold 0.5')
print('\n')
evaluate_multilabel(y_test.values, y_test_pred_tuned, aspect_cols, title='Test - tuned thresholds')
print('\n')
evaluate_multilabel(y_test.values, y_test_pred_tuned_top2, aspect_cols, title='Test - tuned thresholds + top_k=2')

## 7. Ensemble nhieu ClassifierChain random order

Do ket qua ClassifierChain phu thuoc vao thu tu nhan, cell nay train nhieu chain voi cac random order khac nhau va lay trung binh probability.

In [ ]:
chain_seeds = [11, 22, 33, 44, 55]
chain_ensemble = []

for seed in chain_seeds:
    chain_model = build_chain_model(random_state=seed)
    chain_model.fit(X_train, y_train)
    chain_ensemble.append(chain_model)
    print('seed:', seed, 'order:', [aspect_cols[i] for i in chain_model.named_steps['clf'].order_])


def ensemble_predict_proba(models, X_text):
    probas = [model.predict_proba(X_text) for model in models]
    return np.mean(probas, axis=0)


ensemble_val_proba = ensemble_predict_proba(chain_ensemble, X_val)
ensemble_thresholds, ensemble_threshold_report = tune_thresholds(
    y_val.values,
    ensemble_val_proba,
    aspect_cols,
)

display(ensemble_threshold_report)

In [ ]:
ensemble_test_proba = ensemble_predict_proba(chain_ensemble, X_test)

ensemble_test_pred_default = (ensemble_test_proba >= 0.5).astype(int)
ensemble_test_pred_tuned = predict_with_thresholds(ensemble_test_proba, ensemble_thresholds, aspect_cols)
ensemble_test_pred_tuned_top2 = predict_with_thresholds(
    ensemble_test_proba,
    ensemble_thresholds,
    aspect_cols,
    top_k=2,
)

evaluate_multilabel(y_test.values, ensemble_test_pred_default, aspect_cols, title='Ensemble test - default threshold 0.5')
print('\n')
evaluate_multilabel(y_test.values, ensemble_test_pred_tuned, aspect_cols, title='Ensemble test - tuned thresholds')
print('\n')
evaluate_multilabel(y_test.values, ensemble_test_pred_tuned_top2, aspect_cols, title='Ensemble test - tuned thresholds + top_k=2')

## 8. Confusion matrix tung aspect

In [ ]:
best_test_pred = ensemble_test_pred_tuned_top2
mcm = multilabel_confusion_matrix(y_test.values, best_test_pred)

plt.figure(figsize=(9, 11))
for i, aspect in enumerate(aspect_cols):
    plt.subplot(3, 2, i + 1)
    sns.heatmap(
        mcm[i],
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Pred 0', 'Pred 1'],
        yticklabels=['True 0', 'True 1'],
    )
    plt.title(aspect)

plt.tight_layout()
plt.show()

## 9. Xem cac mau du doan sai

In [ ]:
results = pd.DataFrame({
    'review_cleaned': X_test.values,
    'true_aspects': [', '.join(np.array(aspect_cols)[row.astype(bool)]) for row in y_test.values],
    'pred_aspects': [', '.join(np.array(aspect_cols)[row.astype(bool)]) for row in best_test_pred],
})

misclassified = results[results['true_aspects'] != results['pred_aspects']]
print('Misclassified:', len(misclassified), '/', len(results))
display(misclassified.head(20))

## 10. Du doan review moi

In [ ]:
new_reviews = [
    'The food was tasty but the price was too high.',
    'Great ambiance and friendly service.',
    'The staff were slow but the meal was delicious.',
]

new_proba = ensemble_predict_proba(chain_ensemble, new_reviews)
new_pred = predict_with_thresholds(new_proba, ensemble_thresholds, aspect_cols, top_k=2)

for review, pred_row, proba_row in zip(new_reviews, new_pred, new_proba):
    aspects = np.array(aspect_cols)[pred_row.astype(bool)]
    print(review)
    print('Predicted aspects:', ', '.join(aspects))
    print('Probabilities:', dict(zip(aspect_cols, np.round(proba_row, 3))))
    print()